# 01 — Getting Started with AG2 Beta

Build an agent, give it search tools, and use an **observer** to capture every citation the search returns — then render real hyperlinks.

Set `OPENAI_API_KEY` and `EXA_API_KEY` in your `.env`.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolResultsEvent
from autogen.beta.tools import ExaToolkit

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Observer pattern — capture URLs live

`@agent.observer(ToolResultsEvent)` registers a callback (sync or async) that fires every time the agent receives tool results. We use it to record every `title → url` pair the search returns — before the agent even responds.

In [3]:
title_to_url: dict[str, str] = {}

agent = Agent(
    "surveyor",
    prompt="You are a research surveyor. Search broadly, cite real sources.",
    config=config,
    tools=[exa],
)


@agent.observer(ToolResultsEvent)
def capture_urls(event: ToolResultsEvent) -> None:
    """Capture title→url from every search result, live."""
    for r in event.results:
        result = getattr(r, "result", None)
        if result is None:
            continue
        for part in getattr(result, "parts", []):
            data = getattr(part, "data", None)
            if data is None:
                continue
            for hit in getattr(data, "results", None) or []:
                title = getattr(hit, "title", None)
                url = getattr(hit, "url", None)
                if title and url:
                    title_to_url[title] = url

## Ask the agent

We use a shared `MemoryStream` so we can walk the events afterward.

In [4]:
from IPython.display import Markdown, display

stream = MemoryStream()

reply = await agent.ask(
    "Find 5 recent papers on reactive event-driven multi-agent coordination. "
    "For each, give title, year, and one key finding.",
    stream=stream,
)

display(Markdown(reply.body))

Here are 5 recent papers on reactive / event-driven multi-agent coordination, with a brief key finding for each:

| Title | Year | One key finding |
|---|---:|---|
| **Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control** | 2023 | Proposes a model-free MARL method that jointly learns communication and control policies for distributed multi-agent systems, showing event-triggered coordination can be learned from data rather than relying on a precise system model. Source: PMLR / L&DC 2023. |
| **Fixed-time event-triggered control for multi-agent systems with input delay** | 2023 | Shows that fixed-time consensus/formation-like coordination can be achieved for heterogeneous multi-agent systems with input delays while reducing communication and avoiding Zeno behavior. Source: PLOS ONE. |
| **Adaptive Event-Triggered Consensus of Multi-Agent Systems in Sense of Asymptotic Convergence** | 2024 | Introduces an adaptive event-triggered mechanism that reduces unnecessary transmissions while still ensuring consensus convergence under the proposed control design. Source: Sensors (MDPI). |
| **Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus** | 2024 | Combines fixed-time theory with event-triggering to get fast distributed consensus and large communication savings; the paper reports about **79.92%** lower communication cost in simulation. Source: Ain Shams Engineering Journal / ScienceDirect. |
| **Deep reinforcement learning of event-triggered communication and control for multi-agent cooperative transport** | 2021/2025 (conference publication visible in IEEE Xplore) | Learns when to communicate and how to control in a cooperative transport task, demonstrating that event-triggered communication can improve efficiency in multi-agent transport scenarios. Source: IEEE Xplore. |

A couple of notes:
- I prioritized papers that are **clearly about event-triggered or event-driven coordination** in multi-agent systems.
- If you want, I can do a second pass and narrow this to **only robotics papers**, or **only papers from 2024–2025**, or **only learning-based / MARL papers**.

## Captured URLs

The observer fired during the turn. Every search hit is now in `title_to_url`.

In [5]:
lines = [f"**{len(title_to_url)} URLs captured by observer:**", ""]
for title, url in title_to_url.items():
    lines.append(f"- [{title}]({url})")
display(Markdown("\n".join(lines)))

**27 URLs captured by observer:**

- [Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control](https://proceedings.mlr.press/v211/kesper23a.html)
- [Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus](https://www.sciencedirect.com/science/article/pii/S2090447924004866)
- [Fully Distributed Event-Triggered Control of Nonlinear Multiagent Systems Under Directed Graphs: A Model-Free DRL Approach | OpenReview](https://openreview.net/forum?id=P0V53zKRNH)
- [Adaptive Collective Responses to Local Stimuli in Anonymous Dynamic Networks](https://export.arxiv.org/pdf/2304.12771v2.pdf)
- [[PDF] A Visualized Framework for Event Cooperation with Generative Agents](https://ojs.aaai.org/index.php/AAAI/article/view/42386/46347)
- [[PDF] Toward Multi-Agent Reinforcement Learning for Distributed Event ...](https://proceedings.mlr.press/v211/kesper23a/kesper23a.pdf)
- [Event-driven Multi-Agent Concurrent and Collaborative ...](https://dl.acm.org/doi/10.1145/3779232.3779273)
- [Distributed coordination control of multi-agent systems under intermittent sampling and communication: a comprehensive survey | Science China Information Sciences | Springer Nature Link](https://link.springer.com/article/10.1007/s11432-024-4355-1?error=cookies_not_supported&code=9d328597-c000-4102-8b9a-bfd083f89a14)
- [Multi-Agent Coordination across Diverse Applications](https://arxiv.org/abs/2502.14743)
- [A survey on LLM-based multi-agent systems - Springer Nature](https://link.springer.com/article/10.1007/s44336-024-00009-2)
- [A Survey on Context-Aware Multi-Agent Systems: Techniques, Challenges and Future Directions](https://arxiv.org/html/2402.01968v1)
- [Leader–Follower Consensus of Switched Multi-Agent Systems Under Distributed Event-Triggered Scheme](https://www.mdpi.com/2073-8994/17/12/2079)
- [[2604.06813v1] Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation](https://arxiv.org/abs/2604.06813v1)
- [Event-Triggered Control Protocols for Achieving Bipartite Consensus in Switched Multi-Agent Systems](https://www.mdpi.com/2673-4052/7/1/22)
- [Deep reinforcement learning of event-triggered communication and control for multi-agent cooperative transport | IEEE Conference Publication | IEEE Xplore](https://ieeexplore.ieee.org/document/9561274/)
- [Adaptive Event-Triggered Consensus of Multi-Agent Systems in ...](https://www.mdpi.com/1424-8220/24/2/339)
- [Cooperative Fault-Tolerant Control for Heterogeneous Multiagent Systems: A Dual Dynamic Event-Triggered Approach | IEEE Journals & Magazine | IEEE Xplore](https://ieeexplore.ieee.org/document/10982523/)
- [Event-Triggered Collaborative Control Strategies for Multi-agent Unmanned Systems](https://xbzrb.aeu.edu.cn/EN/10.12018/j.issn.2097-0730.20241125005)
- [Dynamic event-triggered consensus of multi-agent systems with ...](https://link.springer.com/article/10.1007/s44443-025-00277-y)
- [Optimal time-varying formation tracking control for nonlinear multi-agent systems via event-triggered and self-triggered reinforcement learning - ADS](https://ui.adsabs.harvard.edu/abs/2025NonDy.114...37P/abstract)
- [Event-driven Multi-Agent Concurrent and Collaborative Coordination](https://dl.acm.org/doi/10.1145/3779232.3779273)
- [Sequential Cooperative Multi-Agent Online Learning and Adaptive Coordination Control in Dynamic and Uncertain Environments[v1] | Preprints.org](https://www.preprints.org/manuscript/202601.0836/v1)
- [CoCoPlan: Adaptive Coordination and Communication for Multi-robot Systems in Dynamic and Unknown Environments](https://arxiv.org/html/2601.10116v1)
- [Event-triggered adaptive consensus for multi-robot task allocation](https://www.sciencedirect.com/science/article/pii/S0140366426000897)
- [[2603.05621v2] RACAS: Controlling Diverse Robots With a Single Agentic System](https://arxiv.org/abs/2603.05621v2)
- [Fixed-time event-triggered control for multi-agent systems with input ...](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0293424)
- [[2601.08129] Emergent Coordination in Multi-Agent Systems via Pressure Fields and Temporal Decay](https://arxiv.org/abs/2601.08129)

## Walking the stream

The stream records every event. Below we render each type with rich formatting — search results show titles with real hyperlinks, answer results show citations.

In [6]:
import json

from autogen.beta.events import (
    ModelRequest,
    ModelResponse,
    ToolCallEvent,
    ToolResultsEvent,
)


def _date(d):
    return f" \u00b7 {d[:10]}" if d else ""


def _trim(s, n):
    if not s:
        return ""
    s = s.replace("\n", " ").strip()
    return s if len(s) <= n else s[:n] + "\u2026"


def render_tool_data(data):
    """Render Exa results using duck typing (no internal imports needed)."""
    # Search response — has .results list of hits
    results = getattr(data, "results", None)
    if isinstance(results, list) and results and hasattr(results[0], "url"):
        lines = [f"_{len(results)} results_"]
        for i, r in enumerate(results, 1):
            title = getattr(r, "title", None) or getattr(r, "url", "")
            url = getattr(r, "url", "")
            date = _date(getattr(r, "published_date", None))
            lines.append(f"  {i}. [{title}]({url}){date}")
            snippet = _trim(getattr(r, "text", None), 180)
            if snippet:
                lines.append(f"     > {snippet}")
        return "\n".join(lines)

    # Answer result — has .answer and .citations
    answer = getattr(data, "answer", None)
    citations = getattr(data, "citations", None)
    if answer and citations:
        body = [answer, "", f"_{len(citations)} citations_"]
        for i, c in enumerate(citations, 1):
            body.append(f"  {i}. [{getattr(c, 'title', '') or c.url}]({c.url})")
            snippet = _trim(getattr(c, "text", None), 120)
            if snippet:
                body.append(f"     > {snippet}")
        return "\n".join(body)

    return f"`{data!r}`"


tool_calls: dict = {}

for ev in await stream.history.get_events():
    if isinstance(ev, ModelRequest):
        display(Markdown(f"**User** &nbsp; {ev.parts[0].content}"))

    elif isinstance(ev, ToolCallEvent):
        args = json.loads(ev.arguments) if isinstance(ev.arguments, str) else ev.arguments
        tool_calls[ev.id] = {"name": ev.name, "args": args}

    elif isinstance(ev, ToolResultsEvent):
        for r in ev.results:
            call = tool_calls.get(r.parent_id, {})
            name = call.get("name", "?")
            args = call.get("args", {})
            arg_str = ", ".join(f"`{k}`={v!r}" for k, v in args.items())
            data = r.result.parts[0].data
            display(
                Markdown(f"&nbsp;&nbsp;**`{name}`** &nbsp; {arg_str}\n\n{render_tool_data(data)}")
            )

    elif isinstance(ev, ModelResponse):
        if ev.content:
            display(Markdown(f"**Model** &nbsp; {ev.content}"))

**User** &nbsp; Find 5 recent papers on reactive event-driven multi-agent coordination. For each, give title, year, and one key finding.

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='recent papers reactive event-driven multi-agent coordination paper 2023 2024 reactive event driven multi-agent coordination'

_10 results_
  1. [Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control](https://proceedings.mlr.press/v211/kesper23a.html) · 2023-06-06
     > Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control   [edit]  # Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control  Luk…
  2. [Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus](https://www.sciencedirect.com/science/article/pii/S2090447924004866) · 2024-12-01
     > Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus - ScienceDirect [Skip to main content](#screen-reader-m…
  3. [Fully Distributed Event-Triggered Control of Nonlinear Multiagent Systems Under Directed Graphs: A Model-Free DRL Approach | OpenReview](https://openreview.net/forum?id=P0V53zKRNH)
     > Fully Distributed Event-Triggered Control of Nonlinear Multiagent Systems Under Directed Graphs: A Model-Free DRL Approach | OpenReview  ## Fully Distributed Event-Triggered Contro…
  4. [https://export.arxiv.org/pdf/2305.08723v1.pdf](https://export.arxiv.org/pdf/2305.08723v1.pdf) · 2023-05-16
     > Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control Lukas Kesper LUKAS.KESPER@RWTH-AACHEN.DE Sebastian Trimpe TRIMPE@DSME.RWTH-AACHEN.DE Institute for…
  5. [Adaptive Collective Responses to Local Stimuli in Anonymous Dynamic Networks](https://export.arxiv.org/pdf/2304.12771v2.pdf) · 2023-04-28
     > Adaptive Collective Responses to Local Stimuli in Anonymous Dynamic Networks Shunhao Oh ! School of Computer Science, Georgia Institute of Technology Dana Randall !  School of Com…
  6. [https://arxiv.org/pdf/2404.18798](https://arxiv.org/pdf/2404.18798)
     > Multi-Agent Synchronization Tasks Rolando Fernandez DEVCOM Army Research Laboratory The University of Texas at Austin Austin, TX, United States rfernandez@utexas.edu Garrett Warnel…
  7. [[PDF] A Visualized Framework for Event Cooperation with Generative Agents](https://ojs.aaai.org/index.php/AAAI/article/view/42386/46347)
     > A Visualized Framework for Event Cooperation with Generative Agents Yuyang Tian 1,2*, Shunqiang Mao 1,3*, Wenchang Gao 1, Lanlan Qiu 1, Tianxing He 4, 1, 5† 1Shanghai Qi Zhi Instit…
  8. [[PDF] Toward Multi-Agent Reinforcement Learning for Distributed Event ...](https://proceedings.mlr.press/v211/kesper23a/kesper23a.pdf)
     > Proceedings of Machine Learning Research vol 211:1–14, 2023 5th Annual Conference on Learning for Dynamics and Control Toward Multi-Agent Reinforcement Learning for Distributed Eve…
  9. [https://arxiv.org/pdf/2411.07039](https://arxiv.org/pdf/2411.07039)
     > LEARNING COLLECTIVE DYNAMICS OF MULTI-AGENT SYSTEMS USING EVENT-BASED VISION Minah Lee, Uday Kamal, and Saibal Mukhopadhyay School of Electrical and Computer Engineering Georgia In…
  10. [https://arxiv.org/pdf/2509.12390](https://arxiv.org/pdf/2509.12390)
     > Distributed Event-Triggered Distance-Based Formation Control for Multi-Agent Systems Evangelos Psomiadis and Panagiotis Tsiotras Abstract— This paper addresses the problem of colla…

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='multi-agent coordination event-driven reactive paper 2022 2023 2024 survey'

_10 results_
  1. [Event-driven Multi-Agent Concurrent and Collaborative ...](https://dl.acm.org/doi/10.1145/3779232.3779273) · 2026-02-05
     > Event-driven Multi-Agent Concurrent and Collaborative Coordination | Proceedings of the 20th International Conference on Virtual Reality Continuum and its Applications in Industry …
  2. [https://export.arxiv.org/pdf/2202.10450v3.pdf](https://export.arxiv.org/pdf/2202.10450v3.pdf) · 2022-08-17
     > arXiv:2202.10450v3 [cs.MA] 16 Aug 2022 A Survey of Ad Hoc Teamwork Research Reuth Mirsky1,2, Ignacio Carlucho3,⋆, Arrasy Rahman3, Elliot Fosong3, William Macke2, Mohan Sridharan4, …
  3. [Distributed coordination control of multi-agent systems under intermittent sampling and communication: a comprehensive survey | Science China Information Sciences | Springer Nature Link](https://link.springer.com/article/10.1007/s11432-024-4355-1?error=cookies_not_supported&code=9d328597-c000-4102-8b9a-bfd083f89a14) · 2025-04-17
     > Distributed coordination control of multi-agent systems under intermittent sampling and communication: a comprehensive survey | Science China Information Sciences | Springer Nature…
  4. [https://arxiv.org/pdf/2502.14743](https://arxiv.org/pdf/2502.14743)
     > arXiv:2502.14743v2 [cs.MA] 21 Feb 2025 Multi-Agent Coordination across Diverse Applications: A Survey LIJUN SUN, Shenzhen Technology University, China YIJUN YANG, Tencent, China QI…
  5. [Multi-Agent Coordination across Diverse Applications](https://arxiv.org/abs/2502.14743)
     > [2502.14743] Multi-Agent Coordination across Diverse Applications: A Survey  # Computer Science > Multiagent Systems  arXiv:2502.14743 (cs)  [Submitted on 20 Feb 2025 (v1), last re…
  6. [https://arxiv.org/pdf/2604.06813](https://arxiv.org/pdf/2604.06813)
     > Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation Fidel Aznara, Mar Pujolaand Álvaro Díeza aDepartment of Computer Science and Artificial Intelligence, University …
  7. [A survey on LLM-based multi-agent systems - Springer Nature](https://link.springer.com/article/10.1007/s44336-024-00009-2) · 2024-10-08
     > A survey on LLM-based multi-agent systems: workflow, infrastructure, and challenges | Vicinagearth | Springer Nature Link  # A survey on LLM-based multi-agent systems: workflow, in…
  8. [https://orbilu.uni.lu/bitstream/10993/66350/1/SOICT__Multiple_Agent__final_.pdf](https://orbilu.uni.lu/bitstream/10993/66350/1/SOICT__Multiple_Agent__final_.pdf) · 2025-11-14
     > A Survey on Challenges and Emerging Frontiers of Multi-Agent Systems Anh Nguyen-Thi-Mai1, Anh Nguyen-Thi-Van1, Bao Doan-Quoc1, Minh Tran-Duc1, Hung Tran1 ⋆, Miroslav Voznak2, Tu Da…
  9. [A Survey on Context-Aware Multi-Agent Systems: Techniques, Challenges and Future Directions](https://arxiv.org/html/2402.01968v1)
     > A Survey on Context-Aware Multi-Agent Systems: Techniques, Challenges and Future Directions  # A Survey on Context-Aware Multi-Agent Systems: Techniques, Challenges and Future Dire…
  10. [https://arxiv.org/pdf/2503.13415](https://arxiv.org/pdf/2503.13415)
     > Highlights A Comprehensive Survey on Multi-Agent Cooperative Decision-Making: Scenarios, Approaches, Challenges and Perspectives Weiqiang Jin, Hongyang Du, Biao Zhao, Xingwu Tian, …

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='event-triggered multi-agent coordination paper recent'

_10 results_
  1. [Leader–Follower Consensus of Switched Multi-Agent Systems Under Distributed Event-Triggered Scheme](https://www.mdpi.com/2073-8994/17/12/2079)
     > Leader–Follower Consensus of Switched Multi-Agent Systems Under Distributed Event-Triggered Scheme  Next Article in Journal  Symmetric Structures in Mock-Lie Algebras: The Quasi-Ce…
  2. [[2604.06813v1] Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation](https://arxiv.org/abs/2604.06813v1)
     > [2604.06813v1] Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation  # Computer Science > Multiagent Systems  arXiv:2604.06813v1 (cs)  [Submitted on 8 Apr 2026]  # Ti…
  3. [Event-Triggered Control Protocols for Achieving Bipartite Consensus in Switched Multi-Agent Systems](https://www.mdpi.com/2673-4052/7/1/22) · 2026-01-21
     > Event-Triggered Control Protocols for Achieving Bipartite Consensus in Switched Multi-Agent Systems  Next Article in Journal  A Nonlinear Disturbance Observer-Based Super-Twisting …
  4. [Deep reinforcement learning of event-triggered communication and control for multi-agent cooperative transport | IEEE Conference Publication | IEEE Xplore](https://ieeexplore.ieee.org/document/9561274/) · 2025-11-08
     > Deep reinforcement learning of event-triggered communication and control for multi-agent cooperative transport | IEEE Conference Publication | IEEE Xplore  --  --  ### IEEE Account…
  5. [Adaptive Event-Triggered Consensus of Multi-Agent Systems in ...](https://www.mdpi.com/1424-8220/24/2/339) · 2024-01-06
     > Adaptive Event-Triggered Consensus of Multi-Agent Systems in Sense of Asymptotic Convergence ** Next Article in Journal [Design, Optimization, and Experimental Evaluation of Slow L…
  6. [Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus](https://www.sciencedirect.com/science/article/pii/S2090447924004866) · 2024-12-01
     > Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus - ScienceDirect [Skip to main content](#screen-reader-m…
  7. [Cooperative Fault-Tolerant Control for Heterogeneous Multiagent Systems: A Dual Dynamic Event-Triggered Approach | IEEE Journals & Magazine | IEEE Xplore](https://ieeexplore.ieee.org/document/10982523/) · 2025-11-09
     > Cooperative Fault-Tolerant Control for Heterogeneous Multiagent Systems: A Dual Dynamic Event-Triggered Approach | IEEE Journals & Magazine | IEEE Xplore  --  --  ### IEEE Account …
  8. [Event-Triggered Collaborative Control Strategies for Multi-agent Unmanned Systems](https://xbzrb.aeu.edu.cn/EN/10.12018/j.issn.2097-0730.20241125005)
     > -- Event-Triggered Collaborative Control Strategies for Multi-agent Unmanned Systems --  #### 模态框（Modal）标题  #### Please choose a citation manager  RIS (ProCite, Reference Manager) …
  9. [Dynamic event-triggered consensus of multi-agent systems with ...](https://link.springer.com/article/10.1007/s44443-025-00277-y) · 2025-10-15
     > Dynamic event-triggered consensus of multi-agent systems with trigger-dependent communication delay | Journal of King Saud University Computer and Information Sciences | Springer N…
  10. [Optimal time-varying formation tracking control for nonlinear multi-agent systems via event-triggered and self-triggered reinforcement learning - ADS](https://ui.adsabs.harvard.edu/abs/2025NonDy.114...37P/abstract)
     > Optimal time-varying formation tracking control for nonlinear multi-agent systems via event-triggered and self-triggered reinforcement learning - ADS   Now on home page     ## ADS …

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='reactive multi-agent coordination event driven robotics paper recent'

_10 results_
  1. [Event-driven Multi-Agent Concurrent and Collaborative Coordination](https://dl.acm.org/doi/10.1145/3779232.3779273) · 2026-02-05
     > Event-driven Multi-Agent Concurrent and Collaborative Coordination | Proceedings of the 20th International Conference on Virtual Reality Continuum and its Applications in Industry …
  2. [[2604.06813v1] Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation](https://arxiv.org/abs/2604.06813v1)
     > [2604.06813v1] Event-Triggered Adaptive Consensus for Multi-Robot Task Allocation  # Computer Science > Multiagent Systems  arXiv:2604.06813v1 (cs)  [Submitted on 8 Apr 2026]  # Ti…
  3. [Sequential Cooperative Multi-Agent Online Learning and Adaptive Coordination Control in Dynamic and Uncertain Environments[v1] | Preprints.org](https://www.preprints.org/manuscript/202601.0836/v1)
     > Sequential Cooperative Multi-Agent Online Learning and Adaptive Coordination Control in Dynamic and Uncertain Environments[v1] | Preprints.org  Cite  Add to My List  Share Comments…
  4. [CoCoPlan: Adaptive Coordination and Communication for Multi-robot Systems in Dynamic and Unknown Environments](https://arxiv.org/html/2601.10116v1)
     > CoCoPlan: Adaptive Coordination and Communication for Multi-robot Systems in Dynamic and Unknown Environments  # CoCoPlan: Adaptive Coordination and Communication for Multi-robot S…
  5. [Event-triggered adaptive consensus for multi-robot task allocation](https://www.sciencedirect.com/science/article/pii/S0140366426000897) · 2026-03-18
     > Skip to main content Skip to article  - View  PDF - Download full issue  Search ScienceDirect  ## Computer Communications  Volume 251, 1 April 2026, 108499  # Event-triggered adapt…
  6. [[2603.05621v2] RACAS: Controlling Diverse Robots With a Single Agentic System](https://arxiv.org/abs/2603.05621v2)
     > [2603.05621v2] RACAS: Controlling Diverse Robots With a Single Agentic System  # Computer Science > Robotics  arXiv:2603.05621v2 (cs)  [Submitted on 5 Mar 2026 (v1), last revised 1…
  7. [Fixed-time event-triggered control for multi-agent systems with input ...](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0293424) · 2023-11-13
     > Fixed-time event-triggered control for multi-agent systems with input delay | PLOS One  Browse Subject Areas  ?  Click through the PLOS taxonomy to find articles in your field.  Fo…
  8. [https://arxiv.org/pdf/2604.11028](https://arxiv.org/pdf/2604.11028)
     > Federated Single-Agent Robotics: Multi-Robot Coordination Without Intra-Robot Multi-Agent Fragmentation∗ Xue Qin1 Simin Luan2 John See3 Cong Yang4,∗ Zhijun Li2,∗ 1School of Softwar…
  9. [[2601.08129] Emergent Coordination in Multi-Agent Systems via Pressure Fields and Temporal Decay](https://arxiv.org/abs/2601.08129)
     > [2601.08129] Emergent Coordination in Multi-Agent Systems via Pressure Fields and Temporal Decay  # Computer Science > Multiagent Systems  arXiv:2601.08129 (cs)  [Submitted on 13 J…
  10. [https://www.arxiv.org/pdf/2510.03592](https://www.arxiv.org/pdf/2510.03592)
     > Noname manuscript No. (will be inserted by the editor) Deep Reinforcement Learning for Multi-Agent Coordination Kehinde O. Aina · Sehoon Ha Received: date / Accepted: date Abstract…

**Model** &nbsp; Here are 5 recent papers on reactive / event-driven multi-agent coordination, with a brief key finding for each:

| Title | Year | One key finding |
|---|---:|---|
| **Toward Multi-Agent Reinforcement Learning for Distributed Event-Triggered Control** | 2023 | Proposes a model-free MARL method that jointly learns communication and control policies for distributed multi-agent systems, showing event-triggered coordination can be learned from data rather than relying on a precise system model. Source: PMLR / L&DC 2023. |
| **Fixed-time event-triggered control for multi-agent systems with input delay** | 2023 | Shows that fixed-time consensus/formation-like coordination can be achieved for heterogeneous multi-agent systems with input delays while reducing communication and avoiding Zeno behavior. Source: PLOS ONE. |
| **Adaptive Event-Triggered Consensus of Multi-Agent Systems in Sense of Asymptotic Convergence** | 2024 | Introduces an adaptive event-triggered mechanism that reduces unnecessary transmissions while still ensuring consensus convergence under the proposed control design. Source: Sensors (MDPI). |
| **Enhancing multi-agent system coordination: Fixed-time and event-triggered control mechanism for robust distributed consensus** | 2024 | Combines fixed-time theory with event-triggering to get fast distributed consensus and large communication savings; the paper reports about **79.92%** lower communication cost in simulation. Source: Ain Shams Engineering Journal / ScienceDirect. |
| **Deep reinforcement learning of event-triggered communication and control for multi-agent cooperative transport** | 2021/2025 (conference publication visible in IEEE Xplore) | Learns when to communicate and how to control in a cooperative transport task, demonstrating that event-triggered communication can improve efficiency in multi-agent transport scenarios. Source: IEEE Xplore. |

A couple of notes:
- I prioritized papers that are **clearly about event-triggered or event-driven coordination** in multi-agent systems.
- If you want, I can do a second pass and narrow this to **only robotics papers**, or **only papers from 2024–2025**, or **only learning-based / MARL papers**.

## Up next

lionag2's agents don't just return text — they emit **typed events** (`FindingEmitted`, `DepthRequested`, etc.) that drive reactive coordination. Tutorial 02 introduces the event system and structured output.